# cuda-kernels W1 on Colab

Runtime → Change runtime type → **GPU**，然后按顺序跑。


In [ ]:
!nvidia-smi
!nvcc --version


In [ ]:
from pathlib import Path
Path('cuda_check.h').write_text('''#pragma once
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

inline void cuda_check(cudaError_t err, const char* file, int line) {
    if (err != cudaSuccess) {
        std::fprintf(stderr, "CUDA error %s:%d: %s\n", file, line, cudaGetErrorString(err));
        std::exit(EXIT_FAILURE);
    }
}

#define CUDA_CHECK(call) cuda_check((call), __FILE__, __LINE__)
''')
Path('vector_add.cu').write_text('''// W1: vector add. Run on Colab/AutoDL with nvcc (MacBook has no NVIDIA GPU).
#include <cstdio>
#include <vector>
#include "cuda_check.h"

__global__ void vector_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int n = 1 << 20;
    const size_t bytes = n * sizeof(float);

    std::vector<float> ha(n), hb(n), hc(n), href(n);
    for (int i = 0; i < n; ++i) {
        ha[i] = 1.0f;
        hb[i] = 2.0f;
        href[i] = 3.0f;
    }

    float *da = nullptr, *db = nullptr, *dc = nullptr;
    CUDA_CHECK(cudaMalloc(&da, bytes));
    CUDA_CHECK(cudaMalloc(&db, bytes));
    CUDA_CHECK(cudaMalloc(&dc, bytes));
    CUDA_CHECK(cudaMemcpy(da, ha.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db, hb.data(), bytes, cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
    vector_add<<<blocks, threads>>>(da, db, dc, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    CUDA_CHECK(cudaMemcpy(hc.data(), dc, bytes, cudaMemcpyDeviceToHost));

    int mismatches = 0;
    for (int i = 0; i < n; ++i) {
        if (hc[i] != href[i]) ++mismatches;
    }

    std::printf("vector_add n=%d blocks=%d threads=%d time=%.3f ms mismatches=%d\n",
                n, blocks, threads, ms, mismatches);

    CUDA_CHECK(cudaFree(da));
    CUDA_CHECK(cudaFree(db));
    CUDA_CHECK(cudaFree(dc));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return mismatches == 0 ? 0 : 1;
}
''')
Path('matrix_add_2d.cu').write_text('''// W1: 2D matrix add with 2D grid/block mapping.
#include <cstdio>
#include <vector>
#include "cuda_check.h"

__global__ void matrix_add_2d(const float* a, const float* b, float* c, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] + b[idx];
    }
}

int main() {
    const int rows = 1024;
    const int cols = 1024;
    const int n = rows * cols;
    const size_t bytes = n * sizeof(float);

    std::vector<float> ha(n), hb(n), hc(n);
    for (int i = 0; i < n; ++i) {
        ha[i] = static_cast<float>(i % 17);
        hb[i] = static_cast<float>(i % 13);
    }

    float *da = nullptr, *db = nullptr, *dc = nullptr;
    CUDA_CHECK(cudaMalloc(&da, bytes));
    CUDA_CHECK(cudaMalloc(&db, bytes));
    CUDA_CHECK(cudaMalloc(&dc, bytes));
    CUDA_CHECK(cudaMemcpy(da, ha.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db, hb.data(), bytes, cudaMemcpyHostToDevice));

    dim3 threads(16, 16);
    dim3 blocks((cols + threads.x - 1) / threads.x, (rows + threads.y - 1) / threads.y);

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
    matrix_add_2d<<<blocks, threads>>>(da, db, dc, rows, cols);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    CUDA_CHECK(cudaMemcpy(hc.data(), dc, bytes, cudaMemcpyDeviceToHost));

    int mismatches = 0;
    for (int i = 0; i < n; ++i) {
        if (hc[i] != ha[i] + hb[i]) ++mismatches;
    }

    std::printf("matrix_add_2d %dx%d grid=(%u,%u) block=(%u,%u) time=%.3f ms mismatches=%d\n",
                rows, cols, blocks.x, blocks.y, threads.x, threads.y, ms, mismatches);

    CUDA_CHECK(cudaFree(da));
    CUDA_CHECK(cudaFree(db));
    CUDA_CHECK(cudaFree(dc));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return mismatches == 0 ? 0 : 1;
}
''')
print('wrote sources')


In [ ]:
!nvcc -O2 -o vector_add vector_add.cu && ./vector_add


In [ ]:
!nvcc -O2 -o matrix_add_2d matrix_add_2d.cu && ./matrix_add_2d


把上面打印的 GPU 型号与 `time=... ms` 填回仓库 README 的「运行记录」表，然后 commit。
